# 0. La caja negra: `src/toolkit/` aplicado sobre datos reales

Este notebook no genera resultados nuevos -- demuestra cada módulo del
toolkit reusable de forma standalone, sobre datos reales ya descargados por
los 4 dominios (`financial_bcch`, `mining_cochilco`, `agriculture_worldbank`,
`consulting_excel_dwh`). El punto es mostrar que las mismas funciones sirven
para cualquier dominio, sin conocer nada sobre él.


## `missing_data.interpolate_within_group`

Interpola gaps DENTRO de cada país, nunca a través de un cambio de país --
sobre el panel agrícola real (Banco Mundial).


In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.toolkit.missing_data import interpolate_within_group, missingness_report

agri = pd.read_csv("../data/processed/agriculture/agriculture_panel_clean.csv")
missingness_report(agri)


,columna,n_nulos,pct_nulos,dtype
0,country_iso3,0,0.0,object
1,country_name,0,0.0,object
2,year,0,0.0,int64
3,cereal_yield_kg_ha,0,0.0,float64
4,arable_land_pct,0,0.0,float64
5,fertilizer_kg_ha,0,0.0,float64
6,agri_land_pct,0,0.0,float64
7,crop_production_index,0,0.0,float64
8,rural_pop_pct,0,0.0,float64
9,irrigated_land_pct,0,0.0,float64


## `outliers.fix_implausible_level_jumps`

El error real encontrado en la UF de `mindicador.cl` (2014-12-29/30): dos
días seguidos corruptos, con valores parecidos ENTRE SÍ, que una comparación
día-a-día no detecta -- se necesita una mediana móvil. Reproducido acá con
los valores reales encontrados.


In [2]:
from src.toolkit.outliers import fix_implausible_level_jumps

demo = pd.DataFrame({
    "fecha": pd.date_range("2014-12-24", periods=10),
    "uf": [24627.10, 24627.10, 24627.10, 24627.10, 608.15, 607.38, 24627.10, 24627.10, 24627.10, 24627.10],
})
fixed, n = fix_implausible_level_jumps(demo, "uf", sort_by="fecha", threshold=0.5)
print(f"{n} valores corregidos")
fixed


2 valores corregidos


,fecha,uf
0,2014-12-24,24627.1
1,2014-12-25,24627.1
2,2014-12-26,24627.1
3,2014-12-27,24627.1
4,2014-12-28,24627.1
5,2014-12-29,24627.1
6,2014-12-30,24627.1
7,2014-12-31,24627.1
8,2015-01-01,24627.1
9,2015-01-02,24627.1


## `excel_cleaning`: encabezados multi-fila y formato ancho -> largo

Sobre un extracto real del WDI (Banco Mundial) -- columnas de año 1960-2024
en una sola fila.


In [3]:
from src.toolkit.excel_cleaning import wide_years_to_long

wdi_wide = pd.read_csv("../data/raw/consulting/wdi_curated_wide.csv").head(3)
long_sample = wide_years_to_long(
    wdi_wide, id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
)
long_sample.dropna(subset=["valor"]).head(8)


,Country Name,Country Code,Indicator Name,Indicator Code,periodo,valor
2,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,186.132432
5,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1961,186.947182
8,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1962,197.408105
11,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1963,225.447007
14,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1964,209.005786
17,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1965,226.883067
20,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1966,240.962194
23,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1967,243.824367


## `text_cleaning`: parseo de moneda y unificación fuzzy de nombres

Ejemplos ilustrativos de la función (no son un hallazgo de dominio, solo
muestran su comportamiento con inputs mixtos, el tipo de mezcla real que
aparece en exports de sistemas distintos).


In [4]:
from src.toolkit.text_cleaning import parse_currency, unify_similar_names

for raw in ["$1,200.50", "1.200,50", "(20.00)", "1234 (p)", "N/A"]:
    print(f"{raw!r:15s} -> {parse_currency(raw)}")

names = pd.Series(["SQM S.A.", "SQM SA", "sqm s.a.", "Albemarle Corp"])
unify_similar_names(names, threshold=80)


'$1,200.50'     -> 1200.5
'1.200,50'      -> 1200.5
'(20.00)'       -> -20.0
'1234 (p)'      -> 1234.0
'N/A'           -> None


0          SQM S.A.
1          SQM S.A.
2          SQM S.A.
3    Albemarle Corp
dtype: object

## `encoding.zscore_scale` + `torch_trainer`

El mismo escalador y el mismo loop de entrenamiento (`train_with_early_stopping`,
piso de 100 épocas) que usan los 4 dominios -- acá sobre un slice pequeño y
real de features financieras, solo para mostrar la mecánica sin repetir el
modelo completo (ya está en `01_financial_bcch.ipynb`).


In [5]:
import torch
from torch import nn
from src.toolkit.encoding import zscore_scale
from src.toolkit.torch_trainer import train_with_early_stopping

fin_features = pd.read_csv("../data/processed/financial/financial_features.csv").head(400)
X_cols = ["dolar_log_return_lag1", "dolar_volatility_5d", "tpm"]
scaled, stats = zscore_scale(fin_features, X_cols)
X = torch.tensor(scaled[X_cols].values, dtype=torch.float32)
y = torch.tensor(fin_features["target_next_return"].values, dtype=torch.float32).view(-1, 1)

model = nn.Sequential(nn.Linear(3, 8), nn.LeakyReLU(0.1), nn.Linear(8, 1))
result = train_with_early_stopping(model, X[:300], y[:300], X[300:], y[300:], loss_fn=nn.MSELoss(), min_epochs=100, max_epochs=150)
print(f"épocas corridas: {result.epochs_run}, mejor época: {result.best_epoch}")


épocas corridas: 150, mejor época: 150


## `excel_io`: el archivo `.xlsx` antes de que sea una tabla

`excel_cleaning` arregla la forma del dato una vez cargado; `excel_io` resuelve
el paso anterior. Inventariar el WDI real (80 MB, 6 hojas) cuesta ~2 segundos
porque se abre en modo `read_only`, y ya deja ver algo que no se sospecha
mirando el archivo en Excel: la hoja más grande del libro no es la de datos.


In [6]:
from src.toolkit.excel_io import (
    coerce_excel_errors, excel_serial_to_datetime, hidden_rows_and_columns,
    inventory_workbook, profile_cell_types, read_sheet_expanding_merges,
)

inventory_workbook("../data/raw/consulting/WDIEXCEL.xlsx")


,hoja,estado,visible,filas,columnas
0,Data,visible,True,401395,69
1,Country,visible,True,266,31
2,Series,visible,True,1510,20
3,country-series,visible,True,8249,3
4,series-time,visible,True,149,3
5,footnote,visible,True,842972,4


### Filas y columnas ocultas: un mismo archivo real usa "oculto" con dos sentidos opuestos

El `.xlsx` de COCHILCO tiene 144 filas ocultas y 1 columna oculta. **Las 144
filas ocultas son todas meses reales** (COCHILCO colapsa el detalle mensual para
dejar a la vista solo los subtotales anuales), así que descartarlas borraría 144
de los 156 meses del archivo. La única columna oculta, en cambio, es
`Chuqui y R.Tomic`: uno de los subtotales disfrazados que el dominio minero ya
excluye por identidad numérica.

Por eso `read_sheet_expanding_merges` no descarta nada por defecto y separa las
dos banderas: una sola bandera aplicada a los dos ejes habría hecho exactamente
lo incorrecto en la mitad de los casos, dentro de un mismo archivo.


In [7]:
from datetime import datetime

COCHILCO = "../data/raw/mining/cochilco_produccion_mensual.xlsx"
HOJA = "Prod.Cu-Mina x Faena (2014+)"

filas_ocultas, columnas_ocultas = hidden_rows_and_columns(COCHILCO, sheet=HOJA)
raw = read_sheet_expanding_merges(COCHILCO, sheet=HOJA)  # sin descartar nada
meses_reales = set(raw.index[raw[0].apply(lambda v: isinstance(v, datetime))])

print(f"filas ocultas: {len(filas_ocultas)}")
print(f"de ellas, meses reales: {len(set(filas_ocultas) & meses_reales)}")
print(f"meses reales que quedan visibles: {len(meses_reales - set(filas_ocultas))}")
print(f"columna oculta: {[raw.iloc[6][c] for c in columnas_ocultas]}")


filas ocultas: 144
de ellas, meses reales: 144
meses reales que quedan visibles: 12
columna oculta: ['Chuqui y R.Tomic']


### Errores de fórmula, números de serie y tipos mezclados

Tres diagnósticos que no existen en un CSV. El ejemplo de literales de error es
ilustrativo (el archivo de COCHILCO no trae fórmulas rotas); el perfil de tipos
sí corre sobre el archivo real, y muestra la mezcla texto/número/fecha que
convierte toda la hoja en `object` antes de limpiarla.


In [8]:
con_errores = pd.DataFrame({
    "produccion": [100, "#DIV/0!", 300, "#N/A"],
    "empresa": ["Codelco", "Escondida", "#REF!", "Candelaria"],
})
limpio, reporte = coerce_excel_errors(con_errores)
print(reporte.to_string(index=False))

# Fechas guardadas como número de serie, con el bug del 29-02-1900 de Excel.
print(excel_serial_to_datetime([45292, 60, 61]).tolist())

profile_cell_types(raw, columns=[0, 1, 2])


   columna  n_errores     literales
produccion          2 #DIV/0!, #N/A
   empresa          1         #REF!
[Timestamp('2024-01-01 00:00:00'), NaT, Timestamp('1900-03-01 00:00:00')]


,columna,dtype_pandas,n_nulos,n_numero,n_texto,n_fecha,n_booleano,n_otro,tipo_dominante,mixta
0,0,object,2316,2,16,156,0,0,fecha,True
1,1,object,2317,169,4,0,0,0,numero,True
2,2,object,2321,163,6,0,0,0,numero,True


## `excel_export`: devolver el dato limpio como entregable, no como `to_excel`

Sobre el panel minero real ya limpio (150 meses x 47 columnas): encabezado
congelado, autofiltro, anchos calculados, formato numérico declarado y una hoja
`Diccionario` que documenta cada columna -- el archivo que efectivamente se
manda, y que un tercero puede auditar sin leer el código que lo generó.


In [9]:
from src.toolkit.excel_export import build_data_dictionary, write_analysis_workbook

panel = pd.read_csv("../data/processed/mining/mining_panel_clean.csv", parse_dates=["fecha"])
formatos = {c: "#,##0.000" for c in panel.columns if c != "fecha"}

ruta = write_analysis_workbook(
    "../data/processed/mining/mining_panel_clean.xlsx",
    {"Panel mensual": panel}, number_formats=formatos,
)
print(f"{ruta} ({ruta.stat().st_size / 1024:.0f} KB)")
build_data_dictionary({"Panel mensual": panel}).head(6)


..\data\processed\mining\mining_panel_clean.xlsx (52 KB)


,hoja,columna,tipo,n_filas,n_no_nulos,pct_faltante,n_unicos,minimo,maximo,ejemplo
0,Panel mensual,fecha,datetime64[ns],150,150,0.0,150,2014-01-01 00:00:00,2026-06-01 00:00:00,2014-01-01 00:00:00
1,Panel mensual,Chuqui y R.Tomic,float64,150,150,0.0,149,32.258,80.03,54.413
2,Panel mensual,Chuquicamata,float64,150,150,0.0,148,10.622,54.039,21.532
3,Panel mensual,Radomiro Tomic,float64,150,150,0.0,150,14.149,35.318,32.881
4,Panel mensual,Ministro Hales,float64,150,150,0.0,137,1.5,24.707,8.5
5,Panel mensual,Salvador,float64,150,150,0.0,116,0.0,11.6,5.2


## `sql_dump`: leer y escribir un dump SQL sin levantar el motor

El mismo panel exportado como dump Postgres y vuelto a leer a DataFrame. El
round-trip es la prueba real: los nombres de columna de COCHILCO incluyen
`Chuqui y R.Tomic` y `Centinela (sulfuros)`, y ambos rompen un parseo ingenuo
(el punto se confunde con el separador `esquema.tabla`, el paréntesis cierra
antes de tiempo la lista de columnas del `INSERT`).


In [10]:
from src.toolkit.sql_dump import dataframe_to_sql_dump, inventory_sql_dump, read_sql_dump

dump = dataframe_to_sql_dump(
    panel, "mining_produccion_mensual",
    "../data/processed/mining/mining_panel_clean.sql", dialect="postgres",
)
print(inventory_sql_dump(dump).to_string(index=False))

vuelta = read_sql_dump(dump)["mining_produccion_mensual"]
vuelta["fecha"] = pd.to_datetime(vuelta["fecha"])
pd.testing.assert_frame_equal(vuelta, panel, check_dtype=False)
print("round-trip idéntico celda a celda")
vuelta[["fecha", "Chuqui y R.Tomic", "Centinela (súlfuros)"]].head(3)


                    tabla  n_filas  n_columnas origen
mining_produccion_mensual      150          47 INSERT
round-trip idéntico celda a celda


,fecha,Chuqui y R.Tomic,Centinela (súlfuros)
0,2014-01-01,54.413,13.4
1,2014-02-01,48.554,13.2
2,2014-03-01,54.002,15.0


## `model_zoo`: las tres familias de modelos que se suman a las tres de siempre

Cada dominio compara ahora **6 modelos sobre el mismo split cronológico**: su
baseline, el MLP, XGBoost, y los tres de `model_zoo` (lineal regularizado,
Random Forest y una LSTM sobre secuencias). Los tres eligen hiperparámetros
mirando el split de validación, nunca con `GridSearchCV`, que baraja las filas
y entrenaría con datos posteriores a los que después evalúa.

Acá se muestran sobre el panel minero real: 95 meses de train, 21 de
validación, 21 de test.


In [11]:
import numpy as np
from src.toolkit.model_zoo import build_sequences, fit_elasticnet, fit_random_forest
from src.domains.mining_cochilco.features import FEATURE_COLUMNS, TARGET_COLUMN
from src.domains.mining_cochilco.model import chronological_split
from src.toolkit.encoding import zscore_scale

minero = pd.read_csv("../data/processed/mining/mining_features.csv", parse_dates=["fecha"])
tr, va, te = chronological_split(minero)
y_tr, y_va, y_te = (d[TARGET_COLUMN].values for d in (tr, va, te))

X_tr, stats = zscore_scale(tr[FEATURE_COLUMNS].reset_index(drop=True), FEATURE_COLUMNS)
def escalar(d):
    out = d[FEATURE_COLUMNS].reset_index(drop=True).copy()
    for c in FEATURE_COLUMNS:
        m, s = stats[c]
        out[c] = (out[c] - m) / s
    return out

lineal = fit_elasticnet(X_tr, y_tr, escalar(va), y_va, escalar(te))
bosque = fit_random_forest(
    tr[FEATURE_COLUMNS], y_tr, va[FEATURE_COLUMNS], y_va, te[FEATURE_COLUMNS],
    feature_names=list(FEATURE_COLUMNS),
)
print("lineal:", {k: v for k, v in lineal.metadata.items() if k != "val_rmse"})
print("bosque:", bosque.metadata["features_mas_importantes"][:3])


lineal: {'alpha_relativo': 0.1, 'alpha_efectivo': 3.154526424072838, 'l1_ratio': 0.0, 'n_coeficientes': 12, 'n_coeficientes_no_nulos': 12}
bosque: [{'feature': 'mes', 'importancia': 0.3301}, {'feature': 'yoy_growth', 'importancia': 0.1278}, {'feature': 'total_rolling_mean_12m', 'importancia': 0.0837}]


### Las secuencias nunca cruzan un límite de grupo

`build_sequences` alinea una secuencia a **cada** fila (rellenando las primeras
de cada serie repitiendo su observación más antigua), para que el conjunto de
test de la LSTM sea exactamente el mismo que el de los otros cinco modelos. Con
`group_column`, la historia de un país nunca toma filas de otro -- la misma
disciplina que `interpolate_within_group` aplica a la imputación.


In [12]:
panel_paises = pd.DataFrame({
    "pais": ["CHL"] * 3 + ["ARG"] * 3,
    "anio": [2000, 2001, 2002] * 2,
    "valor": [1.0, 2.0, 3.0, 10.0, 20.0, 30.0],
})
con_grupo, completas = build_sequences(
    panel_paises, ["valor"], window=3, time_column="anio", group_column="pais")
sin_grupo, _ = build_sequences(panel_paises, ["valor"], window=3, time_column="anio")

print("primera fila de ARG, agrupando :", con_grupo[3].ravel())
print("primera fila de ARG, sin agrupar:", sin_grupo[3].ravel(), "<- contaminada con CHL")
print("filas con historia completa:", completas.tolist())


primera fila de ARG, agrupando : [10. 10. 10.]
primera fila de ARG, sin agrupar: [ 1.  1. 10.] <- contaminada con CHL
filas con historia completa: [False, False, True, False, False, True]


### El resultado real de los 6 modelos, por dominio

Leído directamente de los `metrics.json` que dejó cada pipeline. Vale la pena
mirar las dos filas donde el ganador cambió al sumar los tres modelos nuevos, y
la fila donde el modelo más sofisticado queda último.


In [13]:
import json

filas = []
for dominio, archivo in [
    ("financiero", "financial"), ("minería", "mining"),
    ("agricultura", "agriculture"), ("consultoría", "consulting"),
]:
    resultados = json.loads(open(f"../outputs/{archivo}/metrics.json", encoding="utf-8").read())["results"]
    fila = {"dominio": dominio}
    fila.update({nombre: round(m["r2"], 4) for nombre, m in resultados.items()})
    fila["mejor"] = max(resultados, key=lambda n: resultados[n]["r2"])
    filas.append(fila)

pd.DataFrame(filas).set_index("dominio")


,baseline_media,mlp_pytorch,xgboost,elasticnet,random_forest,lstm,mejor,baseline_estacional,baseline_media_pais
dominio,,,,,,,,,
financiero,-0.0012,0.0040,-0.0021,-0.0023,-0.0032,-0.0083,mlp_pytorch,NaN,NaN
minería,NaN,0.2515,0.5146,0.2396,0.4793,0.0260,xgboost,0.1289,NaN
agricultura,NaN,0.8724,0.8839,0.8422,0.9044,0.7990,random_forest,NaN,-0.3271
consultoría,-1.6075,0.9222,0.9382,0.9278,0.9426,0.9367,random_forest,NaN,NaN


## `drift`: por qué los baselines de "predecir el promedio" dan R² negativo

Un R² negativo del baseline se ve como un error de cálculo y no lo es: es
**drift del target**. `target_shift` lo mide antes de entrenar nada, y el R² que
predice a partir solo del desplazamiento de la media coincide con el R² real que
después reporta el baseline en `metrics.json`.


In [14]:
import json
from src.toolkit.drift import drift_report, target_shift
from src.domains.consulting_excel_dwh.features import FEATURE_COLUMNS as CON_F, TARGET_COLUMN as CON_T
from src.domains.consulting_excel_dwh.model import chronological_split as con_split

consultoria = pd.read_csv("../data/processed/consulting/consulting_features.csv")
tr_c, va_c, te_c = con_split(consultoria)

desplazamiento = target_shift(tr_c[CON_T], te_c[CON_T])
r2_baseline = json.load(open("../outputs/consulting/metrics.json", encoding="utf-8"))["results"]["baseline_media"]["r2"]

print(f"media del target: {desplazamiento['media_expected']:.2f} -> {desplazamiento['media_actual']:.2f} años")
print(f"desplazamiento: {desplazamiento['desplazamiento_en_desvios']:+.2f} desvíos | PSI {desplazamiento['psi']:.3f}")
print(f"R² que implica ese desplazamiento : {desplazamiento['r2_de_predecir_la_media_vieja']:.4f}")
print(f"R² real del baseline en metrics.json: {r2_baseline:.4f}")


media del target: 62.47 -> 72.02 años
desplazamiento: +0.85 desvíos | PSI 1.361
R² que implica ese desplazamiento : -1.6075
R² real del baseline en metrics.json: -1.6075


El reporte por columna ordena por severidad, porque la pregunta operativa no es
"¿hay drift?" (casi siempre hay algo) sino "¿qué columna miro primero?". PSI y
KS van juntos a propósito: KS trae p-valor pero con muestras grandes marca como
significativo cualquier movimiento minúsculo; el PSI no tiene p-valor pero mide
la magnitud real del desplazamiento.


In [15]:
reporte = drift_report(tr_c, te_c, columns=list(CON_F))
print(reporte["veredicto"].value_counts().to_string())
reporte.head(5)[["columna", "psi", "veredicto", "ks_pvalor", "media_expected", "media_actual"]]


veredicto
severo      15
moderado     5


,columna,psi,veredicto,ks_pvalor,media_expected,media_actual
0,pib_per_capita_usd_lag1,2.146459,severo,4.256652e-124,4054.709028,14079.883460
1,mortalidad_infantil,1.867035,severo,3.760046e-120,60.427436,20.731656
2,mortalidad_infantil_lag1,1.810852,severo,8.366462e-121,61.819571,21.282086
3,pib_per_capita_usd,1.394686,severo,1.929185e-120,4279.470242,14528.108459
4,esperanza_vida_lag1,1.345798,severo,1.135749e-78,61.840762,71.650075


## `keys`: la cardinalidad real de un join, antes de ejecutarlo

`pd.merge` no lanza ninguna excepción cuando la clave derecha está duplicada:
multiplica filas en silencio, y todo agregado posterior queda inflado. Acá, sobre
el warehouse WDI real, la búsqueda de claves candidatas **redescubre el grano de
la tabla de hechos desde el dato**, sin mirar el DDL.


In [16]:
import duckdb
from src.toolkit.keys import describe_join, find_candidate_keys, find_orphans

con = duckdb.connect("../data/processed/consulting/wdi_warehouse.duckdb", read_only=True)
hechos = con.execute("select * from fact_indicator_value").fetchdf()
dim_pais = con.execute("select * from dim_country").fetchdf()
con.close()

print("grano de la tabla de hechos:", find_candidate_keys(hechos, max_columns=3))
print("claves de dim_country     :", find_candidate_keys(dim_pais, max_columns=1))
pd.Series(describe_join(hechos, dim_pais, on="country_code"))


grano de la tabla de hechos: [('country_code', 'indicator_code', 'anio')]
claves de dim_country     : [('country_code',), ('nombre_pais',)]


cardinalidad                N:1
clave_izquierda_unica     False
clave_derecha_unica        True
filas_izquierda          132600
filas_derecha               217
huerfanas_izquierda           0
huerfanas_derecha             0
filas_tras_left_join     132600
filas_tras_inner_join    132600
factor_multiplicacion       1.0
dtype: object

Contra la dimensión **cruda** del Excel (265 filas, con agregados regionales
incluidos), las filas huérfanas del lado de la dimensión son exactamente los
agregados que `clean.py` descartó -- un número que sale del join, sin volver a
aplicar el filtro.


In [17]:
crudo = pd.read_csv("../data/raw/consulting/wdi_country_dim.csv").rename(columns={"Country Code": "country_code"})
_sin_pais, dim_sin_uso = find_orphans(hechos, crudo[["country_code"]].drop_duplicates(), on="country_code")

print(f"paises en la dimension cruda : {crudo['country_code'].nunique()}")
print(f"paises reales en el warehouse: {len(dim_pais)}")
print(f"filas de dimension sin ningun hecho asociado: {len(dim_sin_uso)}")


paises en la dimension cruda : 265
paises reales en el warehouse: 217
filas de dimension sin ningun hecho asociado: 48


## `leakage`: el chequeo que hay que correr contra el resultado *bueno*

La fuga de target es el único error de este proyecto cuyo síntoma es un
resultado bueno: un R² alto no dispara ninguna alarma. El Random Forest de
consultoría pone el 97,2% de su importancia en una sola feature -- una firma
numérica idéntica a la de una fuga. El chequeo la marca, pero como `revisar` y
no como `fuga`, porque no hay ninguna relación determinística con el target: la
esperanza de vida del año en curso está genuinamente disponible al predecir la
del siguiente.


In [18]:
from src.toolkit.leakage import leakage_report

fuga = leakage_report(tr_c[CON_F], tr_c[CON_T], X_test=te_c[CON_F])
print("veredicto:", fuga["veredicto"])
print("sospechosas:", fuga["features_sospechosas"])
print("relaciones exactas con el target:", fuga["relaciones_exactas"])
print("filas compartidas entre train y test:", fuga["solapamiento_train_test"]["n_solapadas"])
fuga["poder_individual"].head(4)


veredicto: revisar
sospechosas: ['esperanza_vida', 'esperanza_vida_lag1']
relaciones exactas con el target: []
filas compartidas entre train y test: 0


,feature,r2_individual,correlacion,sospechosa
0,esperanza_vida,0.984957,0.992450,True
1,esperanza_vida_lag1,0.978050,0.988964,True
2,mortalidad_infantil,0.865049,-0.930080,False
3,mortalidad_infantil_lag1,0.858548,-0.926579,False


Y el contraste, sobre una fuga fabricada a propósito (el único dato sintético de
este notebook, marcado como tal): una columna que **es** el target en otras
unidades. Acá el veredicto sí es `fuga`, porque la evidencia es determinística y
no una correlación alta.


In [19]:
from src.toolkit.leakage import exact_relations

y_demo = tr_c[CON_T].to_numpy()[:500]
X_demo = pd.DataFrame({
    "driver_legitimo": tr_c["gasto_salud_pct_pib"].to_numpy()[:500],
    "target_en_meses": y_demo * 12,   # el target, en otras unidades
})
print(exact_relations(X_demo, y_demo).to_string(index=False))
print("veredicto:", leakage_report(X_demo, y_demo)["veredicto"])


        feature relacion  constante
target_en_meses escalada   0.083333
veredicto: fuga


## Conclusión

Las mismas ~45 funciones de `src/toolkit/` (limpieza, outliers, texto, Excel,
dumps SQL, encoding, calidad de datos, visualización, entrenamiento y modelos)
se reusan sin cambios en los 4 dominios -- lo único que cambia entre dominios es
el dato de entrada real y la interpretación del resultado, nunca la técnica.
